# Wolfram Alpha LLM API를 Claude의 도구로 사용하기
이 레시피에서는 Wolfram Alpha LLM API를 Claude가 사용할 도구로 연동하는 방법을 보여 줍니다. Claude가 Wolfram Alpha API에 질의를 보내 계산된 응답을 받고, 그것으로 사용자 질문에 답할 수 있게 됩니다.

## 1단계: 환경 설정
먼저 필요한 라이브러리를 설치하고 Claude API 클라이언트를 설정합니다. WolframAlpha를 사용하려면 APP ID도 설정해야 합니다. [여기](https://developer.wolframalpha.com/access)에서 무료로 가입하고 이 프로젝트용 App ID를 새로 만들 수 있습니다.

In [ ]:
import json
import urllib.parse

import requests
from anthropic import Anthropic

client = Anthropic()

# Replace 'YOUR_APP_ID' with your actual Wolfram Alpha AppID
WOLFRAM_APP_ID = "YOUR_APP_ID"
MODEL_NAME = "claude-haiku-4-5"

## 2단계: Wolfram Alpha LLM API 도구 정의하기
Claude가 Wolfram Alpha LLM API에 질의를 보내 계산된 응답을 받을 수 있게 하는 도구를 정의합니다.

In [ ]:
import urllib.parse

import requests


def wolfram_alpha_query(query):
    # URL-encode the query
    encoded_query = urllib.parse.quote(query)

    # Make a request to the Wolfram Alpha LLM API
    url = (
        f"https://www.wolframalpha.com/api/v1/llm-api?input={encoded_query}&appid={WOLFRAM_APP_ID}"
    )
    response = requests.get(url, timeout=30)

    if response.status_code == 200:
        return response.text
    else:
        return f"Error: {response.status_code}: {response.text}"


tools = [
    {
        "name": "wolfram_alpha",
        "description": "A tool that allows querying the Wolfram Alpha knowledge base. Useful for mathematical calculations, scientific data, and general knowledge questions.",
        "input_schema": {
            "type": "object",
            "properties": {
                "search_query": {
                    "type": "string",
                    "description": "The query to send to the Wolfram Alpha API.",
                }
            },
            "required": ["query"],
        },
    }
]

이 코드에서는 질의를 입력으로 받아 URL 인코딩한 뒤, 제공된 AppID로 Wolfram Alpha LLM API에 요청을 보내는 wolfram_alpha_query 함수를 정의합니다. 요청이 성공하면 API가 계산한 응답을, 문제가 있으면 오류 메시지를 반환합니다.

그런 다음 문자열 타입의 query 속성 하나를 받는 입력 스키마로 wolfram_alpha 도구를 정의합니다.

## 3단계: Claude와 상호작용하기
이제 Claude가 Wolfram Alpha 도구와 상호작용해 사용자 질문에 답하는 과정을 살펴보겠습니다.

In [27]:
def process_tool_call(tool_name, tool_input):
    if tool_name == "wolfram_alpha":
        return wolfram_alpha_query(tool_input["search_query"])


def chat_with_claude(user_message):
    print(f"\n{'=' * 50}\nUser Message: {user_message}\n{'=' * 50}")
    prompt = f"""Here is a question: {user_message}. Please use the Wolfram Alpha tool to answer it. Do not reflect on the quality of the returned search results in your response."""

    message = client.beta.tools.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        messages=[{"role": "user", "content": prompt}],
    )

    print("\nInitial Response:")
    print(f"Stop Reason: {message.stop_reason}")
    print(f"Content: {message.content}")

    if message.stop_reason == "tool_use":
        tool_use = next(block for block in message.content if block.type == "tool_use")
        tool_name = tool_use.name
        tool_input = tool_use.input

        print(f"\nTool Used: {tool_name}")
        print("Tool Input:")
        print(json.dumps(tool_input, indent=2))

        tool_result = process_tool_call(tool_name, tool_input)

        print("\nTool Result:")
        print(str(json.dumps(tool_result, indent=2)))

        response = client.beta.tools.messages.create(
            model=MODEL_NAME,
            max_tokens=2000,
            tools=tools,
            messages=[
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": message.content},
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_use.id,
                            "content": str(tool_result),
                        }
                    ],
                },
            ],
        )

        print("\nResponse:")
        print(f"Stop Reason: {response.stop_reason}")
        print(f"Content: {response.content}")
    else:
        response = message

    final_response = None
    for block in response.content:
        if hasattr(block, "text"):
            final_response = block.text
            break

    print(f"\nFinal Response: {final_response}")

    return final_response

## 4단계: 직접 사용해 보기!
이제 Claude가 Wolfram Alpha를 쓸 수 있게 되었으니, 예시 질문을 몇 개 던져 보겠습니다.

In [28]:
# Example usage
print(chat_with_claude("What are the 5 largest countries in the world by population?"))
print(chat_with_claude("Calculate the square root of 1764."))
print(chat_with_claude("What is the distance between Earth and Mars?"))


User Message: What are the 5 largest countries in the world by population?

Initial Response:
Stop Reason: tool_use
Content: [ContentBlock(text='<thinking>\nThe query "What are the 5 largest countries in the world by population?" can be answered well using the wolfram_alpha tool, which has knowledge about countries and populations. The search_query parameter is the only required parameter, and the query text provided by the user can be used directly as the search_query value without any additional information needed. \n</thinking>', type='text'), ContentBlockToolUse(id='toolu_01VCQ5xAzMNdyXYsepbSAJLY', input={'search_query': 'What are the 5 largest countries in the world by population?'}, name='wolfram_alpha', type='tool_use')]

Tool Used: wolfram_alpha
Tool Input:
{
  "search_query": "What are the 5 largest countries in the world by population?"
}

Tool Result:
"Query:\n\"What are the 5 largest countries in the world by population?\"\n\nInput interpretation:\n5 largest countries | by